# One digitising zone: fast zone-only model and farm map (Kaggle)

Give it an area drawn in Google Earth Pro (a KMZ in `data/zones/`) and the Planet scenes over it, and it maps the
farms in that area in about 1–3 hours instead of 12. It works **only on the zone**:

1. the zone's own Planet imagery (usually 2 dates) and the farms already digitised inside it;
2. **fine-tunes existing models on the zone's farms** instead of training from scratch: the round-2 network
   (trained on all 14,800 Laikipia farms), at 3 m and with the image **enlarged ×2** (small plots), and the
   published **Fields of the Planet** network (pretrained on 3 m PlanetScope fields);
3. compares them on held-out zone farms with the round-2 model **as it is**; a fine-tuned model is used only if it
   is better by `SELECT_MARGIN`;
4. maps the zone: every date is predicted and **averaged**, polygons are built once with outlines that cannot
   cross, the farms already digitised are **kept as they are** and model farms are added only where they are new;
5. writes `zone/kmz/<zone>_farms.kmz` (Google Earth Pro, one folder per work tile; an area drawn without tiles
   gets a 3 km grid T01, T02, ...) and `report.html`.

Zones in the repository: `salama_area.kmz` (137 km², dataset `salama-area-planetscope`, about 1,240 digitised
farms inside) and `mutara_path.kmz` (Mutara – Aberdares path, dataset `mutara-zone-planetscope`, about 200
digitised farms: use `SELECT_MARGIN = 0.05` there, its validation set is tiny).

**Inputs:** *Add Input → Your Work → Datasets →* the zone's private dataset (the zone imagery + the round-2
model). Code, farm polygons and the zone file come from GitHub.
**Settings:** *GPU T4 x2* (or P100), *Internet On*. Run with **Save Version → Save & Run All (Commit)**.

## 1. Settings

In [ ]:
from pathlib import Path

DATASET_NAME = "salama-area-planetscope"   # zone imagery + round-2 model (name, user/name or full path)
REPO = "https://github.com/mangaorphy/Farm_Digitisation.git"
BRANCH = "main"
PROJECT_SUBDIR = "farm_boundary_detection"
ZONE_FILE = "data/zones/salama_area.kmz"   # in the repository: the zone polygon (+ optional work tiles)
ZONE_NAME = "salama"                       # used in the output file names
SELECT_MARGIN = 0.02      # a fine-tuned model must beat the round-2 model by this much object F1
                          # (0.05 for a zone with few digitised farms, e.g. the Mutara path)

QUICK_TEST = False        # True = a few minutes' dry run on a few patches and one tile
TIME_BUDGET_H = 4.0       # whole notebook
EPOCHS = 40
PATIENCE = 8
USE_TTA = True            # flip averaging for validation, test and the map
ENSEMBLE_TOP = 2          # also try the average of the 2 best models (0 = off)
SPLIT_BLOCKS_M = [3000, 2000, 1500, 1000]   # split block sizes tried in this order (see step 4)
MAP_TILES = None          # None = every tile; or e.g. ["T02", "T03"]
MIN_COMPACT = 0.15        # model polygons less compact than this (long slivers, ragged blobs) are dropped
MAX_FARM_HA = None        # model polygons larger than this are dropped; None = the 99th percentile of the
                          # zone's digitised farms (at least 5 ha). Digitised farms are never dropped.

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
import time
T0 = time.time()

## 2. Install and get the code

In [ ]:
import os, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rasterio", "geopandas", "pyogrio", "scikit-image",
                "segmentation-models-pytorch", "shapely>=2.1"],
               check=True)
code_dir = WORK / "Farm_Digitisation"
if not code_dir.exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, str(code_dir)], check=True)
else:
    subprocess.run(["git", "-C", str(code_dir), "pull", "--ff-only"], check=False)
PROJECT = code_dir / PROJECT_SUBDIR
sys.path.insert(0, str(PROJECT))
os.chdir(PROJECT)
import torch, src
print("torch", torch.__version__, "| GPUs:",
      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] or "NONE - turn on the GPU!")

## 3. Zone imagery, zone farms and configuration

Only the farms within the zone (+ 500 m) are used, written to a small labels file.

In [ ]:
import json, shutil, logging, zipfile
import pandas as pd, geopandas as gpd
from IPython.display import Image, display
from src.utils.helpers import load_config, setup_logging
from src.models.multi_task_unet import load_checkpoint
from src.data.ingestion import load_farm_polygons
from src.gis.zone import read_zone

def find_dataset(name):
    p = Path(name)
    if p.is_absolute() and p.is_dir():
        return p
    hits = sorted((d for d in INPUT.rglob("*") if d.is_dir() and d.name == p.name), key=lambda d: len(d.parts))
    if hits:
        return hits[0]
    sr = sorted(INPUT.rglob("*AnalyticMS_SR*.tif"))
    return sr[0].parent if sr else None

DATA = find_dataset(DATASET_NAME)
assert DATA, f"dataset {DATASET_NAME!r} not found under {INPUT}: attach it with Add Input"
tifs = sorted(p for p in DATA.rglob("*.tif") if "udm" not in p.name.lower())
models_found = sorted(DATA.rglob("*best_model.pth")) or sorted(INPUT.rglob("*best_model.pth"))
assert tifs, f"no scenes in {DATA}"
assert models_found, "the round-2 model (*best_model.pth) is not in the dataset"
BASE_MODEL = models_found[0]
mk = load_checkpoint(BASE_MODEL, "cpu")["model_kwargs"]
print(f"{len(tifs)} scenes in {DATA}:", *[t.name for t in tifs], sep="\n  ")
print("round-2 model:", BASE_MODEL.name, {k: mk[k] for k in ("encoder", "decoder_channels", "distance_head")})

setup_logging(WORK / "logs", "INFO", "zone")
REPORT = WORK / "report"; REPORT.mkdir(parents=True, exist_ok=True)

# --- zone farms only
zone, tiles = read_zone(PROJECT / ZONE_FILE)
utm = zone.estimate_utm_crs()
labels_all, _ = load_farm_polygons(load_config(PROJECT / "configs" / "config.yaml"))
area = zone.to_crs(utm).buffer(500).iloc[0]
zl = labels_all.to_crs(utm)
zl = zl[zl.intersects(area)]
keep_cols = [c for c in ("FARM_ID", "SOURCE", "QA_FLAG", "SCHEME", "name") if c in zl]
ZONE_LABELS = WORK / "zone_labels"; shutil.rmtree(ZONE_LABELS, ignore_errors=True); ZONE_LABELS.mkdir()
zl[keep_cols + ["geometry"]].to_crs(4326).to_file(ZONE_LABELS / "zone_farms.gpkg", driver="GPKG")
print(f"{len(zl)} digitised farms in the zone (+500 m); median {zl.area.median() / 1e4:.2f} ha; "
      f"by tile:", gpd.sjoin(zl.representative_point().to_frame("geometry").set_crs(utm),
                             tiles.to_crs(utm), predicate="within").tile.value_counts().sort_index().to_dict())

if MAX_FARM_HA is None:
    MAX_FARM_HA = round(max(5.0, float(zl.area.quantile(0.99)) / 1e4), 1)
print(f"model polygons above {MAX_FARM_HA} ha or below compactness {MIN_COMPACT} will be dropped "
      "(bush blobs); digitised farms are always kept")

QUICK = ["train.max_train_patches=24", "train.epochs=2", "evaluation.n_visuals=2"] if QUICK_TEST else []
BLOCK_M = SPLIT_BLOCKS_M[0]           # set in step 4
BASE = [f"paths.imagery_dir={DATA}", f"paths.labels_dir={ZONE_LABELS}",
        "split.fractions={train: 0.6, val: 0.2, test: 0.2}",
        "patches.size=256", "patches.stride=128", "patches.eval_stride=256", "patches.min_farm_pixels=150",
        "inference.patch_size=512", "inference.overlap=128", "inference.batch_size=4",
        f"model.encoder={mk['encoder']}", f"model.decoder_channels={list(mk['decoder_channels'])}",
        f"model.distance_head={str(bool(mk.get('distance_head'))).lower()}",
        f"model.full_resolution_branch={str(bool(mk['full_resolution_branch'])).lower()}",
        f"model.psp_pooling={str(bool(mk['psp_pooling'])).lower()}",
        f"model.condition_interior_on_boundary={str(bool(mk['condition_interior_on_boundary'])).lower()}",
        "train.num_workers=4", f"train.epochs={EPOCHS}", f"train.patience={PATIENCE}"]

def make_cfg(exp=None, extra=()):
    exp_dir = WORK / "experiments" / (exp or "_none")
    over = BASE + [f"split.block_size_m={BLOCK_M}", f"paths.processed_dir={WORK / 'prep' / f'b{BLOCK_M}'}", f"paths.models_dir={exp_dir / 'models'}",
                   f"paths.outputs_dir={exp_dir / 'outputs'}", f"paths.logs_dir={WORK / 'logs'}", *extra, *QUICK]
    return load_config(PROJECT / "configs" / "config.yaml", over)

def hours_left():
    return TIME_BUDGET_H - (time.time() - T0) / 3600

# experiment -> overrides; "E0" = the round-2 model as it is (no training). Simplest first (wins ties).
FT = [f"train.init_from={BASE_MODEL}", "train.lr=0.0001", "train.warmup_epochs=1"]
EXPERIMENTS = {
    "E0_round2_as_is": None,
    "F1_finetune":     FT + ["train.batch_size=16"],
    "F2_finetune_x2":  FT + ["model.input_upsample=2", "train.batch_size=6"],
    # published Fields of the Planet network (3 m PlanetScope fields), fine-tuned on the zone's farms
    "F3_ftp_finetune": ["model.arch=ftp_unet", "normalization.mode=scale", "train.lr=0.0002", "train.warmup_epochs=1",
                        "augmentation.gain=0.3", "augmentation.gamma=0.15", "train.batch_size=16"],
}

## 4. Preprocess the zone (masks, split, patches, checks)

Train / validation / test are whole ground blocks, so nearby farms never sit on both sides. The block size that
works depends on how the digitised farms are spread: large blocks lose fewer patches on block edges, small blocks
are needed when the farms sit in a few clusters. Each size in `SPLIT_BLOCKS_M` is tried until validation and test
both get at least `MIN_EVAL_PATCHES` patches.

In [ ]:
from src.pipeline import prepare
MIN_EVAL_PATCHES = 2 if QUICK_TEST else 6
tried = {}
for BLOCK_M in SPLIT_BLOCKS_M:
    cfg0 = make_cfg()
    meta = Path(cfg0.paths.processed_dir) / "metadata"
    try:
        if not ((meta / "patches.csv").exists() and (meta / "validation_report.json").exists()):
            prepare(cfg0)
        n = pd.read_csv(meta / "patches.csv").split.value_counts().reindex(["train", "val", "test"]).fillna(0).astype(int)
    except Exception as e:  # e.g. a split without patches
        print(f"block {BLOCK_M} m: not usable ({str(e)[:90]})")
        shutil.rmtree(cfg0.paths.processed_dir, ignore_errors=True); continue
    tried[BLOCK_M] = n.to_dict(); print(f"block {BLOCK_M} m: patches {tried[BLOCK_M]}")
    if min(n["val"], n["test"]) >= MIN_EVAL_PATCHES:
        break
assert tried, "no block size gave train, validation and test patches: add more digitised farms to the zone"
BLOCK_M = max(tried, key=lambda b: (min(tried[b]["val"], tried[b]["test"]) >= MIN_EVAL_PATCHES,
                                    min(tried[b]["val"], tried[b]["test"])))
for b in tried:
    if b != BLOCK_M:
        shutil.rmtree(WORK / "prep" / f"b{b}", ignore_errors=True)
cfg0 = make_cfg()
meta = Path(cfg0.paths.processed_dir) / "metadata"
print(f"using {BLOCK_M} m blocks")
patches = pd.read_csv(meta / "patches.csv")
patch_summary = patches.groupby("split").agg(patches=("patch_id", "size"), farms=("n_farms", "sum"),
                                              farm_px=("farm_pixels", "sum"), boundary_px=("boundary_pixels", "sum"))
checks = pd.DataFrame(json.loads((meta / "validation_report.json").read_text()))[["check", "ok", "message"]]
display(patch_summary, checks)
vis = Path(cfg0.paths.processed_dir) / "visualizations"
mask_figs = sorted(vis.glob("*_masks.png"))
for f in mask_figs[:2] + [vis / "split_map.png"]:
    if f.exists():
        display(Image(filename=str(f), width=1300))
print(f"{hours_left():.1f} h left")

## 5. Fine-tune (starts from the round-2 weights)

In [ ]:
from src.training.trainer import train
from src.utils.visualization import plot_history, plot_training_curves

best_pt = {"E0_round2_as_is": BASE_MODEL}
to_train = [e for e, v in EXPERIMENTS.items() if v is not None]
for k, exp in enumerate(to_train):
    budget = (hours_left() - 1.0) / (len(to_train) - k)     # keep 1 h for comparison and the map
    if budget < 0.2 and not QUICK_TEST:
        print(f"{exp}: skipped - no time left"); continue
    cfg = make_cfg(exp, EXPERIMENTS[exp] + [f"train.max_hours={max(budget, 0.2):.2f}"])
    print(f"\n========== {exp} ({budget:.1f} h max): {EXPERIMENTS[exp]} ==========")
    best_pt[exp] = train(cfg, resume=True)
    plot_history(Path(cfg.paths.models_dir) / "training_history.csv", Path(cfg.paths.models_dir) / "training_history.png")

hist = {e: WORK / "experiments" / e / "models" / "training_history.csv" for e in best_pt if e != "E0_round2_as_is"}
curves_png = plot_training_curves(hist, REPORT / "training_curves.png") if hist else None
if curves_png:
    display(Image(filename=str(curves_png), width=1400))
train_summary = pd.DataFrame({e: {"epochs run": len(pd.read_csv(h)), "best epoch": int(pd.read_csv(h).val_score.idxmax()),
                                  "best val_score": pd.read_csv(h).val_score.max(),
                                  "minutes": pd.read_csv(h).minutes.sum()} for e, h in hist.items()}).T
display(train_summary)
print(f"{hours_left():.1f} h left")

## 6. Compare on held-out zone farms (validation)

Each model predicts the validation patches (with TTA), its polygon post-processing is tuned on them, and it is
scored on **individual farms** (object F1: one predicted polygon per farm, IoU ≥ 0.5).

In [ ]:
from src.training.evaluation import predict_patches, tune_postprocess, evaluate, ensemble_caches, true_areas
from src.utils.visualization import plot_comparison

AREAS = true_areas(cfg0)
SMALL = dict(stage1={"seed_source": ["boundary", "edt"], "seed_threshold": [0.4, 0.6], "min_seed_px": [25]},
             stage2={"interior_threshold": [0.4], "boundary_threshold": [0.5], "min_area_m2": [200]}) if QUICK_TEST else {}
caches, tuned, rows = {}, {}, {}

def score(name, cache):
    pc, _ = tune_postprocess(cfg0, cache, SMALL.get("stage1"), SMALL.get("stage2"))
    tuned[name] = pc
    res = evaluate(cfg0, None, "val", WORK / "evaluation" / name, postprocess=pc, cache=cache, areas=AREAS, title=name)
    rows[name] = {k: res[k] for k in ("object_f1", "object_precision", "object_recall", "mean_matched_iou",
                                      "merged_predictions", "split_farms", "interior_iou", "boundary_f1", "farms_true")}
    rows[name].update({f"pp_{k}": pc[k] for k in ("seed_source", "seed_threshold", "min_seed_px", "interior_threshold", "min_area_m2")})

for exp, m in best_pt.items():
    caches[exp] = predict_patches(cfg0, m, "val", tta=USE_TTA)
    score(exp, caches[exp])
singles = pd.DataFrame(rows).T.sort_values("object_f1", ascending=False)
# an ensemble needs models that read the image the same way: the best model and others of its family
family = lambda e: "ftp" if "model.arch=ftp_unet" in (EXPERIMENTS.get(e) or []) else "resunet"
same = [e for e in singles.index if family(e) == family(singles.index[0])]
ENS = same[:ENSEMBLE_TOP] if ENSEMBLE_TOP > 1 and len(same) > 1 else []
if ENS:
    score("ENSEMBLE", ensemble_caches([caches[e] for e in ENS]))
comparison = pd.DataFrame(rows).T.loc[[e for e in list(best_pt) + ["ENSEMBLE"] if e in rows]]
comparison.to_csv(WORK / "comparison_val.csv")

f1 = comparison["object_f1"].astype(float)
top = float(f1[[e for e in best_pt]].max())
MARGIN = SELECT_MARGIN
BEST = next(e for e in best_pt if float(f1[e]) >= top - MARGIN)
if "ENSEMBLE" in f1 and float(f1["ENSEMBLE"]) > float(f1[BEST]) + MARGIN:
    BEST = "ENSEMBLE"
BEST_MODELS = [best_pt[e] for e in ENS] if BEST == "ENSEMBLE" else best_pt[BEST]
print(f"BEST = {BEST}" + (f" {ENS}" if BEST == "ENSEMBLE" else ""), f"| val object F1 {float(f1[BEST]):.3f} "
      f"(round-2 model as it is: {float(f1['E0_round2_as_is']):.3f})")
comp_png = plot_comparison(comparison, REPORT / "comparison_val.png")
display(Image(filename=str(comp_png), width=1100))
for c in caches.values():
    c["items"].clear()
comparison

## 7. Test the winner once (held-out zone blocks)

In [ ]:
from src.utils.visualization import plot_object_outcomes
pp = tuned[BEST]
test = evaluate(cfg0, None, "test", WORK / "evaluation" / "test", postprocess=pp,
                cache=predict_patches(cfg0, BEST_MODELS, "test", tta=USE_TTA), areas=AREAS, title=f"{BEST} (test)")
base_test = evaluate(cfg0, None, "test", WORK / "evaluation" / "test_round2_as_is", postprocess=tuned["E0_round2_as_is"],
                     cache=predict_patches(cfg0, BASE_MODEL, "test", tta=USE_TTA), areas=AREAS,
                     title="round-2 model as it is (test)") if BEST != "E0_round2_as_is" else test
keys = ["object_f1", "object_precision", "object_recall", "mean_matched_iou", "merged_predictions", "split_farms",
        "interior_iou", "boundary_f1", "farms_true"]
test_table = pd.DataFrame({f"{BEST} (test)": {k: test[k] for k in keys},
                           "round-2 as it is (test)": {k: base_test[k] for k in keys}})
display(test_table)
tdir = WORK / "evaluation" / "test"
outcomes_png = plot_object_outcomes(test, REPORT / "test_outcomes.png", f"{BEST}: test")
display(pd.read_csv(tdir / "by_size.csv").set_index("size"))
for f in (tdir / "by_size.png", tdir / "confusion.png", outcomes_png):
    display(Image(filename=str(f), width=1100))

## 8. Map the zone

Every scene over the zone is predicted (both dates), the probabilities are averaged, polygons are built once,
and the result is merged with the farms already digitised: **magenta** = digitised (kept as is), **green** =
model, likely farm, **orange** = model, check first.

In [ ]:
from src.gis.zone import run_zone
cfg_map = make_cfg(BEST if BEST in EXPERIMENTS else None,
                   (EXPERIMENTS.get(ENS[0] if BEST == "ENSEMBLE" else BEST) or []))
cfg_map.postprocess = pp
cfg_map.inference.tta = USE_TTA
tiles_to_map = (["T02"] if QUICK_TEST else MAP_TILES)
zres = run_zone(cfg_map, BEST_MODELS, PROJECT / ZONE_FILE, tifs, WORK / "zone", ZONE_NAME, digitised=labels_all,
                only_tiles=tiles_to_map, max_area_ha=MAX_FARM_HA, min_compact=MIN_COMPACT)
farms = zres["farms"]
per_tile = zres["counts"]
display(per_tile)
model_f = farms[farms.source == "Model"]
print(f"{len(farms)} farms: {int((farms.source == 'Digitized').sum())} digitised (kept) + {len(model_f)} from the model "
      f"({int((model_f.review == 'likely farm').sum())} likely, {int((model_f.review == 'check').sum())} to check); "
      f"images used: {zres['images']}")
print({k: str(v) for k, v in zres["paths"].items()})

In [ ]:
import numpy as np, rasterio, matplotlib.pyplot as plt
from rasterio.windows import from_bounds
from src.utils.visualization import stretch
preview_png = REPORT / "map_preview.png"
busy = farms.tile.value_counts()
busy = busy[busy.index.str.match(r"T\d+")]
if len(busy):
    t_name = busy.index[0]
    best = None                                          # the scene with most data around the tile centre
    for t in tifs:
        with rasterio.open(t) as s:
            c = tiles[tiles.tile == t_name].to_crs(s.crs).geometry.iloc[0].centroid
            b = (c.x - 1500, c.y - 1500, c.x + 1500, c.y + 1500)
            arr = s.read([6, 4, 2] if s.count == 8 else [3, 2, 1], window=from_bounds(*b, s.transform),
                         boundless=True, fill_value=0)
            valid = float((arr.max(0) > 0).mean())
            if best is None or valid > best[0]:
                best = (valid, arr, b, s.crs, t.name)
    valid, rgb, b, crs, img_name = best
    f = farms.to_crs(crs).cx[b[0]:b[2], b[1]:b[3]]
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(stretch(rgb), extent=(b[0], b[2], b[1], b[3]))
    for src_, rev, col in (("Digitized", "existing", "magenta"), ("Model", "likely farm", "lime"), ("Model", "check", "orange")):
        g = f[(f.source == src_) & (f.review == rev)]
        if len(g):
            g.boundary.plot(ax=ax, color=col, linewidth=0.9)
    ax.set_xlim(b[0], b[2]); ax.set_ylim(b[1], b[3]); ax.axis("off")
    ax.set_title(f"{t_name}, 3 x 3 km on {img_name[:8]}: magenta digitised, green model likely, orange model check")
    fig.savefig(preview_png, dpi=100, bbox_inches="tight"); plt.show()

## 9. Report and download

In [ ]:
from src.utils.report import build_report
vt = WORK / "evaluation" / BEST if BEST in rows else WORK / "evaluation" / "ENSEMBLE"
summary = {"best": BEST, "ensemble_members": ENS if BEST == "ENSEMBLE" else [], "tta": USE_TTA,
           "postprocess": dict(pp), "val": rows[BEST], "test": test, "test_round2_as_is": base_test,
           "images": zres["images"], "farms": int(len(farms)), "model_farms": int(len(model_f))}
(WORK / "summary.json").write_text(json.dumps(summary, indent=2, default=str))
for m in (BEST_MODELS if isinstance(BEST_MODELS, list) else [BEST_MODELS]):
    tag = Path(m).parent.parent.name if Path(m).parent.name == "models" else "E0_round2_as_is"
    shutil.copy(m, WORK / "zone" / f"{tag}_best_model.pth")
report = build_report(WORK / "report.html", f"{ZONE_NAME}: zone model and farm map", [
    ("Summary", [f"Winner: <b>{BEST}</b>. Test object F1 <b>{test['object_f1']:.3f}</b> vs the round-2 model as it "
                 f"is {base_test['object_f1']:.3f} (about {test['farms_true']} test farms: treat small differences as "
                 f"noise). Map: {len(farms)} farms = {int((farms.source == 'Digitized').sum())} digitised + "
                 f"{len(model_f)} from the model.", test_table]),
    ("Zone data", [patch_summary, checks] + mask_figs[:2]),
    ("Training curves", [train_summary] + ([curves_png] if curves_png else [])),
    ("Validation comparison", [comparison, comp_png]),
    ("Winner: test by farm size and confusion matrix", [pd.read_csv(tdir / "by_size.csv").set_index("size"),
                                                        tdir / "by_size.png", tdir / "confusion.png", outcomes_png]),
    ("Map: farms per tile", [per_tile] + ([preview_png] if preview_png.exists() else [])),
])
with zipfile.ZipFile(WORK / f"{ZONE_NAME}_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in [WORK / "report.html", WORK / "summary.json", WORK / "comparison_val.csv"] + \
             [p for p in (WORK / "zone").rglob("*") if p.is_file() and "prediction_masks" not in p.parts]:
        z.write(f, f.relative_to(WORK))
print("report ->", report, f"| zip -> {ZONE_NAME}_results.zip |", f"{hours_left():.1f} h of budget left")

**Download** from the version's *Output* tab: `zone/kmz/<zone>_farms.kmz` (open in Google Earth Pro: a folder
per tile; magenta = already digitised, green = model likely farm, orange = check), `report.html`, or everything in
`<zone>_results.zip` (KMZ, GeoPackage, Shapefile, GeoJSON, model, summary). The fused probability raster is in
`zone/prediction_masks/<zone>_fused_prob.tif` (QGIS).